# Adding a New Potential in Python

galpy makes it straightforward to add new gravitational potentials.
Once implemented, a new potential automatically works with all of galpy's
orbit integration, action-angle, and distribution function machinery.

The steps are:

1. Create a class that inherits from `galpy.potential.Potential`
2. Implement `__init__` calling `Potential.__init__`
3. Implement `_evaluate(self, R, z, phi=0., t=0.)` for the potential value
4. Implement `_Rforce(self, R, z, phi=0., t=0.)` for the radial force
5. Implement `_zforce(self, R, z, phi=0., t=0.)` for the vertical force
6. Optionally implement `_dens`, `_R2deriv`, `_z2deriv`, `_Rzderiv`, etc.

In [ ]:
%matplotlib inline
import numpy
import matplotlib.pyplot as plt
from galpy.potential import Potential
from galpy.orbit import Orbit

## Example: a custom softened power-law potential

Let's implement a softened isothermal-like potential of the form:

$$\Phi(R, z) = \frac{1}{2}\,v_0^2\,\ln(R^2 + z^2 + d^2)$$

where $v_0$ sets the circular velocity scale and $d$ is a softening length.
This is similar to the logarithmic potential but implemented from scratch
to illustrate the process.

In [ ]:
class SoftenedLogPotential(Potential):
    """A softened logarithmic potential: Phi = 0.5 * v0^2 * ln(R^2 + z^2 + d^2)."""

    def __init__(self, amp=1.0, v0=1.0, d=0.1, ro=None, vo=None):
        """
        Initialize the potential.

        Parameters
        ----------
        amp : float
            Overall amplitude.
        v0 : float
            Velocity scale (in natural units).
        d : float
            Softening length (in natural units).
        ro, vo : float, optional
            Unit conversion parameters.
        """
        Potential.__init__(self, amp=amp, ro=ro, vo=vo)
        self._v0 = v0
        self._d = d
        self._d2 = d**2.0

    def _evaluate(self, R, z, phi=0.0, t=0.0):
        return 0.5 * self._v0**2.0 * numpy.log(R**2.0 + z**2.0 + self._d2)

    def _Rforce(self, R, z, phi=0.0, t=0.0):
        return -(self._v0**2.0) * R / (R**2.0 + z**2.0 + self._d2)

    def _zforce(self, R, z, phi=0.0, t=0.0):
        return -(self._v0**2.0) * z / (R**2.0 + z**2.0 + self._d2)

## Using the new potential

Our new potential immediately works with galpy's plotting functions,
orbit integration, and other tools.

In [ ]:
sp = SoftenedLogPotential(v0=1.0, d=0.1)
sp.plotRotcurve(Rrange=[0.01, 5.0], grid=1001)

## Orbit integration with the new potential

In [ ]:
o = Orbit([1.0, 0.1, 1.1, 0.0, 0.1])
ts = numpy.linspace(0, 100, 10000)
o.integrate(ts, sp)
o.plot()

## Using normalize

The `normalize` keyword can be used to set the circular velocity at R=1 to a
desired value. We can call `normalize()` after initialization.

In [ ]:
sp_norm = SoftenedLogPotential(v0=1.0, d=0.1)
sp_norm.normalize(1.0)
print("v_circ at R=1:", sp_norm.vcirc(1.0))

## Physical units with amp_units

To allow users to specify the amplitude of your potential in physical units
(e.g., solar masses or km/s), add an `amp_units` class attribute. For example,
for a potential whose amplitude has units of velocity squared:

```python
class MyPotential(Potential):
    amp_units = 'velocity2'
    ...
```

Valid options include `'mass'`, `'velocity2'`, `'density'`, and `'surfacedensity'`.
This enables the potential to accept astropy Quantity inputs for the amplitude.

See the galpy documentation on physical units for full details.

## Optional methods

For better performance (especially in action-angle calculations), you can
also implement:

- `_dens(self, R, z, phi=0., t=0.)`: the density (via Poisson's equation, galpy
  can compute this numerically, but an analytical form is faster)
- `_R2deriv(self, R, z, phi=0., t=0.)`: second derivative with respect to R
- `_z2deriv(self, R, z, phi=0., t=0.)`: second derivative with respect to z
- `_Rzderiv(self, R, z, phi=0., t=0.)`: mixed second derivative
- `_phitorque(self, R, z, phi=0., t=0.)`: azimuthal torque (non-axisymmetric potentials)

For adding a C implementation for fast orbit integration, see the
"Adding a New Potential in C" tutorial.